<a href="https://colab.research.google.com/github/meltemtaner/phys405/blob/main/Copy_of_Lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PHYS 405 Computerized Data Analysis · Week 2 · Measures of central tendency and dispersion
Instructor: Prof. Dr. Bora Işıldak · Department of Physics, Yeditepe University

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/Phys_405_2026_2027_Fall/blob/main/Labs/Labs2/Lab2.ipynb)

**File → Save a copy in Drive, then work in your copy.** Your work lives in your Drive copy, not in the link.

How the lab works (eight problems, about 14 minutes each, in pairs):
1. Read the problem. Write your prediction in the **Predict** cell.
2. Run the code, then **Compare** with your prediction.
3. Run the checks.
4. Do the **Your turn** task in the empty cell.

Today's physics: pendulum periods, a liquid-nitrogen bath, a pixel detector's time-over-threshold, a resistor, a laser rangefinder, and Week 1's channel spectrum with a small bump.
Problems 1–2 build histograms; Problems 7–8 are the two spectrum problems of Week 1's lab.
The lab uses new data: the lecture's numbers do not carry over, so make your predictions fresh.

**Your Drive folder.** The next cell connects Google Drive (Colab asks for permission once) and makes the folder
`MyDrive/PHY405_Labs/Week02` if it does not exist. Every file this notebook writes goes there, so it is still in your Drive after the Colab machine is gone.

In [1]:
import os
try:
    from google.colab import drive                      # running in Colab
    drive.mount("/content/drive")
    LAB_DIR = "/content/drive/MyDrive/PHY405_Labs/Week02"
except ImportError:                                     # running elsewhere: a local folder
    LAB_DIR = os.path.join(os.getcwd(), "PHY405_Labs", "Week02")
os.makedirs(LAB_DIR, exist_ok=True)
os.chdir(LAB_DIR)                                       # files written below land here
print("working folder:", os.getcwd())

MessageError: Error: credential propagation was unsuccessful

In [3]:
import sys, math, numpy as np, matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'   # sharp figures on high-resolution screens
print("Python", sys.version.split()[0], "· NumPy", np.__version__)

Python 3.13.16 · NumPy 2.1.3


## Problem 1 · Thirty pendulum periods: make their histogram
A stopwatch gives 30 periods of a pendulum (simulated, s): 1.99, 2.02, 2.03, 1.97, 2.00, … (all 30 are in the code below).
A **histogram** cuts the time axis into intervals called **bins** and counts how many readings fall into each bin.
Sort the 30 periods into bins of 0.05 s from 1.90 s to 2.10 s, first by hand on paper, then with `np.histogram`. How many values fall into each bin?

*Given:* the bins are [1.90, 1.95), [1.95, 2.00), [2.00, 2.05), [2.05, 2.10); a value that sits exactly on an edge goes into the bin on its right; the counts must add up to 30.

**Predict.** The four counts, one per bin. Where does a reading of 2.00 s go? Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ([2, 10, 14, 4], "[2.00, 2.05]")

In [ ]:
t = np.array([1.99, 2.02, 2.03, 1.97, 2.00, 2.02, 2.04, 2.02, 2.05, 1.97,
              2.08, 2.05, 1.98, 2.03, 2.02, 1.94, 2.03, 1.98, 2.00, 1.98,
              1.99, 2.01, 2.01, 1.94, 2.00, 1.95, 1.96, 1.99, 2.04, 2.03])  # s
counts, edges = np.histogram(t, bins=[1.90, 1.95, 2.00, 2.05, 2.10])
print(counts, counts.sum())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(edges[:-1], counts, width=0.05, align="edge", edgecolor="k", alpha=0.6)   # one bar per bin
for x0, n in zip(edges[:-1], counts):
    ax.text(x0 + 0.025, n + 0.3, str(n), ha="center")                            # the count on top of each bar
ax.set_xticks(edges); ax.set_ylim(0, counts.max() + 2)
ax.set_xlabel("period (s)"); ax.set_ylabel("number of readings"); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: count: every reading is in exactly one bin, so the counts add up to 30
print(f"{' + '.join(str(n) for n in counts)} = {counts.sum()}")
assert counts.sum() == len(t) == 30
# Check B: independent method: count by hand with the rule  lower edge <= x < upper edge
hand = [int(((t >= lo) & (t < hi)).sum()) for lo, hi in zip(edges[:-1], edges[1:])]
print("by hand:", hand, " np.histogram:", counts.tolist())
assert hand == counts.tolist()
# trap: the readings that sit exactly on an edge, counted in both neighbouring bins
on_edge = np.isin(t, edges[1:-1]).sum()
print(f"trap: {on_edge} readings sit on an edge; counting them twice gives {counts.sum() + on_edge} values out of 30")

**Your turn.** Plot the histogram with `plt.hist`, label both axes with units, and add a title. Which bin is the tallest, and what does that tell you about the period?

*Hint:* `plt.hist(t, bins=[1.90, 1.95, 2.00, 2.05, 2.10])` · `plt.xlabel("period (s)")`, `plt.ylabel("number of readings")`.

In [ ]:
import matplotlib.pyplot as plt

plt.hist(t, bins=[1.90, 1.95, 2.00, 2.05, 2.10], edgecolor='black')

plt.xlabel("period (s)")
plt.ylabel("number of readings")
plt.title("Histogram of Pendulum Periods")

plt.show()

## Problem 2 · The same 30 periods in bins of 0.01, 0.05 and 0.25 s: which picture is right?
Plot the same 30 periods with bins of 0.01 s, 0.05 s and 0.25 s (from 1.80 to 2.25 s). Which bin width gives a useful picture?
Run Problem 1's cell first: it makes the array `t`.

*Given:* `np.round(np.arange(1.80, 2.25 + w/2, w), 2)` makes the bin edges for width `w` (rounding keeps edges such as 2.00 exact); rule of thumb: about √N bins over the range of the data.

**Predict.** Which bin width shows the shape best: 0.01, 0.05 or 0.25 s? Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = 0.05

In [ ]:
for w in (0.01, 0.05, 0.25):
    counts, edges = np.histogram(t, bins=np.round(np.arange(1.80, 2.25 + w / 2, w), 2))
    print(w, len(counts), (counts > 0).sum(), counts.max())

The printout shows, for each bin width: the width (s), the number of bins, how many bins are not empty, and the count in the tallest bin.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
for ax, w in zip(axs, (0.01, 0.05, 0.25)):
    ax.hist(t, bins=np.round(np.arange(1.80, 2.25 + w / 2, w), 2), edgecolor="k", alpha=0.6)
    ax.set_title(f"bin width {w} s"); ax.set_xlabel("period (s)")
axs[0].set_ylabel("number of readings"); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: rule of thumb: about sqrt(30) bins over the range of the data
print(f"sqrt(30) = {math.sqrt(30):.1f} bins over {t.max() - t.min():.2f} s -> width about {(t.max() - t.min()) / math.sqrt(30):.3f} s")
# Check B: count: each of the three histograms still holds all 30 readings
for w in (0.01, 0.05, 0.25):
    counts, edges = np.histogram(t, bins=np.round(np.arange(1.80, 2.25 + w / 2, w), 2))
    print(f"width {w} s: total {counts.sum()}")
    assert counts.sum() == 30

**Your turn.** Make the histogram with a bin width of 0.025 s (edges from 1.925 to 2.10 s). Is the shape still clear? How many bins are non-empty?

*Hint:* `np.round(np.arange(1.925, 2.10 + 0.0125, 0.025), 3)` · count the non-empty bins with `(counts > 0).sum()`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

edges = np.round(np.arange(1.925, 2.10 + 0.0125, 0.025), 3)

counts, edges = np.histogram(t, bins=edges)
plt.hist(t, bins=edges, edgecolor='black')
plt.xlabel("period (s)")
plt.ylabel("number of readings")
plt.title("Bin width = 0.025 s")
plt.show()

non_empty_bins = (counts > 0).sum()

print("Counts in bins:", counts)
print("Number of non-empty bins:", non_empty_bins)

## Problem 3 · A liquid-nitrogen bath reads 104 K on average. Is it boiling?
A Pt100 platinum thermometer in a liquid-nitrogen bath logs 77.3, 77.4, 77.2, 77.5, 77.3, 293.1, 77.4, 77.3 K (illustrative; at reading 6 the logger's multiplexer read the neighbouring room-temperature channel). What is the bath temperature?

*Given:* nitrogen boils at 77.36 K at 1 atm (NIST); the mean and the median; a channel mix-up reads room temperature.

**Predict.** The bath temperature (K): which number do you quote? Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = 77.35

In [ ]:
T = np.array([77.3, 77.4, 77.2, 77.5, 77.3, 293.1, 77.4, 77.3])   # K
print(round(T.mean(), 2), np.median(T))
print(round(T[T < 200].mean(), 2))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 2.8), width_ratios=[2, 1])
for ax, lim in zip(axs, ((70, 300), (77.1, 77.6))):          # full range, and a zoom near 77 K
    ax.plot(T, np.zeros_like(T), "o", ms=8, alpha=0.4, label="readings")
    ax.axvline(T.mean(), c="C3", label=f"mean {T.mean():.2f} K")
    ax.axvline(np.median(T), c="C2", ls="--", label=f"median {np.median(T):.2f} K")
    ax.axvline(77.36, c="gray", ls=":", label="N$_2$ boiling point 77.36 K")
    ax.set_xlim(*lim); ax.set_yticks([]); ax.set_xlabel("T (K)")
axs[1].set_title("zoom: 77.1–77.6 K"); axs[0].legend(loc="upper center", fontsize=8); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: physics: the median agrees with the boiling point of nitrogen
print(f"median {np.median(T):.2f} K vs boiling point 77.36 K")
assert abs(np.median(T) - 77.36) < 0.05
# Check B: independent method: remove the flagged reading (channel mix-up) and take the mean
print(f"mean without the glitch {T[T < 200].mean():.2f} K; the glitch moved the mean by (293.1 - 77.34)/8 = {(293.1 - 77.34) / 8:.1f} K")
assert abs(T[T < 200].mean() - np.median(T)) < 0.02

**Your turn.** A ninth reading also comes out at 293.1 K. What are the mean and the median now? How many glitches can the median tolerate in nine readings?

*Hint:* sort the nine readings; the median is the 5th.

In [ ]:
import numpy as np

T_9 = np.append(T, 293.1)

mean_9 = np.mean(T_9)
median_9 = np.median(T_9)

print(f"New Mean: {mean_9:.2f} K")
print(f"New Median: {median_9:.2f} K")

## Problem 4 · A pixel's time-over-threshold: which number is typical?
A pixel detector records the time-over-threshold of 5000 hits (simulated, skewed with a long tail). Compute the mode, the median and the mean. Which one describes a typical hit?

*Given:* mode = centre of the tallest 5 ns bin; median; mean; a long tail to large values. Sketch the histogram on paper before you run.

**Predict.** Mode, median and mean (ns), in which order? Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = [46.7, 60.0, 68.0]

In [ ]:
rng = np.random.default_rng(3)
t = rng.lognormal(np.log(60), 0.5, 5000)                # ns (simulated)
n, e = np.histogram(t, bins=np.arange(0, 400, 5))
print(e[np.argmax(n)] + 2.5, round(np.median(t), 1), round(t.mean(), 1))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.stairs(n, e, fill=True, alpha=0.5)
for val, lab, c in ((e[np.argmax(n)] + 2.5, "mode", "C1"), (np.median(t), "median", "C2"), (t.mean(), "mean", "C3")):
    ax.axvline(val, c=c, label=f"{lab} {val:.1f} ns")
ax.set_xlim(0, 250); ax.set_xlabel("time over threshold (ns)"); ax.set_ylabel("hits per 5 ns"); ax.legend(); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: ordering: a long right tail requires mode < median < mean
mode = e[np.argmax(n)] + 2.5
assert mode < np.median(t) < t.mean()
print("mode < median < mean: ok")
# Check B: count: the fraction of hits below the mean is not 0.50
print(f"fraction below the mean = {np.mean(t < t.mean()):.2f}")
# the mode depends on the bin width (always state it); the ordering does not
for w in (4, 10):
    nw, ew = np.histogram(t, bins=np.arange(0, 400, w))
    print(f"{w} ns bins: mode = {ew[np.argmax(nw)] + w / 2:.1f} ns")

**Your turn.** Remove the 10 % longest hits and recompute the mean and the median. Which one changes more, and why?

*Hint:* sort the values and keep the first 4500.

In [ ]:
# TODO: write your answer here

## Problem 5 · Five resistor readings: divide by 5 or by 4?
A resistor reads 100.2, 99.8, 100.1, 99.9, 100.5 Ω (illustrative). Compute the standard deviation by hand, once with $1/n$ and once with $1/(n-1)$. Which one estimates the spread of the meter?

*Given:* deviations from the mean; the sum of squares; $s^2 = \sum_i (x_i - \bar x)^2/(n-1)$ for a sample; a simulation with known spread σ = 0.27 Ω.

**Predict.** s with 1/n and with 1/(n − 1) (Ω). Which one is honest? Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
R = np.array([100.2, 99.8, 100.1, 99.9, 100.5])        # ohm
d = R - R.mean()
print(round((d**2).sum(), 3), round(np.std(R, ddof=1), 4), round(np.std(R), 4))
rng = np.random.default_rng(7)
V = np.var(rng.normal(100, 0.27, (100_000, 5)), axis=1, ddof=1)
print(round(V.mean(), 4), round(0.27**2, 4))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(np.arange(1, 6), d, color="C0"); ax.axhline(0, c="k", lw=0.8)
ax.set_xlabel("reading"); ax.set_ylabel("deviation from 100.10 Ω (Ω)")
ax.set_title(f"sum of squares {(d**2).sum():.3f} Ω²: split into 4 or into 5?"); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: ratio: the two versions differ by sqrt(n/(n-1)) = sqrt(5/4)
ratio = np.std(R, ddof=1) / np.std(R)
print(f"ratio {ratio:.3f} = sqrt(5/4) = {math.sqrt(5 / 4):.3f}")
assert abs(ratio - math.sqrt(5 / 4)) < 1e-12
# Check B: simulation: 1/(n-1) averages to the true variance, 1/n is low by (n-1)/n
print(f"true sigma^2 = {0.27**2:.4f}; mean 1/(n-1) variance = {V.mean():.4f}; mean 1/n variance = {V.mean() * 4 / 5:.4f}")

**Your turn.** Only the first two readings exist: 100.2 and 99.8 Ω. Compute s with 1/(n − 1) and with 1/n. How large is the difference?

*Hint:* mean 100.0; two deviations ±0.2.

In [ ]:
# TODO: write your answer here

## Problem 6 · 25 rangefinder readings: how well is the distance known?
A laser rangefinder gives 25 readings of the same distance (simulated). Compute the mean, the standard deviation $s$ and the uncertainty of the mean.

*Given:* $s$ with $1/(n-1)$; standard error $s/\sqrt{n}$; $n = 25$.

**Predict.** The uncertainty of the mean distance (mm). Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
rng = np.random.default_rng(4)
d = np.round(rng.normal(12.346, 0.008, 25), 3)          # m (simulated readings)
s = np.std(d, ddof=1)
print(round(d.mean(), 4), round(s * 1000, 2), round(s / 5 * 1000, 2))

In [ ]:
k = np.arange(1, 26)
run = np.cumsum(d) / k                                  # running mean after k readings
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(k, (d - d.mean()) * 1000, "o", alpha=0.5, label="readings")
ax.fill_between(k, (run - d.mean() - s / np.sqrt(k)) * 1000, (run - d.mean() + s / np.sqrt(k)) * 1000,
                alpha=0.3, color="C1", label="running mean ± s/√k")
ax.axhspan(-s * 1000, s * 1000, color="gray", alpha=0.1, label=f"± s = {s * 1000:.2f} mm")
ax.set_xlabel("number of readings k"); ax.set_ylabel(f"distance − {d.mean():.4f} m (mm)"); ax.legend(fontsize=8); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: scaling: four times the readings halves the error (100 readings -> s/10)
print(f"100 readings: {s / 10 * 1000:.2f} mm (half of {s / 5 * 1000:.2f} mm)")
# Check B: units and digits: the error 1.7 mm fixes the last digit at 0.1 mm
print(f"distance = {d.mean():.4f} ± {s / 5:.4f} m;  trap: s = {s * 1000:.1f} mm as the error of the mean (5x too large)")
assert round(s / 5 * 1000, 1) == 1.7

**Your turn.** How many readings are needed for an uncertainty of the mean of 0.5 mm with the same rangefinder?

*Hint:* s/√n = 0.5 mm; square both sides.

In [ ]:
# TODO: write your answer here

## Problem 7 · A spectrum in channels: what is the unknown peak?
A detector spectrum (simulated, Week 1's) is stored in ADC channels, not keV. It shows Cs-137 (661.657 keV), Co-60 (1173.228 and 1332.492 keV) and one unknown peak. Find each peak's channel from the histogram, turn channels into keV with two known lines, and name the unknown.

*Given:* peak channel = centre of the tallest 8-channel bin in a window; a straight line $E = a \cdot \text{ch} + b$ through two known points; candidate lines: K-40 at 1460.8 keV, Bi-214 at 609.3 keV.

First run the spectrum cell below (its code is hidden, because it contains the answer). It makes `counts, edges`: a histogram with bins of 8 channels.

In [ ]:
# @title Run this cell: it makes the spectrum `counts, edges` (8-channel bins) and plots it
_rng = np.random.default_rng(2028)
_lines = [661.657, 1173.228, 1332.492, 1460.82]
_ev = [_rng.normal(E, 0.06 * 662 / 2.3548 * math.sqrt(E / 662), n) for E, n in zip(_lines, (6000, 3000, 3000, 3000))]
_ev.append(_rng.exponential(300, 5000))
_ch = (np.concatenate(_ev) + 60.0) / 0.75
counts, edges = np.histogram(_ch, bins=np.arange(0, 4096 + 8, 8))       # 8-channel bins
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.stairs(counts, edges, fill=True, alpha=0.6)
ax.set_xlim(0, 2600); ax.set_xlabel("channel"); ax.set_ylabel("counts per 8 channels"); plt.show()

**Predict.** The energy of the unknown peak (keV). Is one known line enough to find it? Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
pk = lambda lo, hi: edges[lo//8 + np.argmax(counts[lo//8:hi//8])] + 4
c_cs, c_co, c_x = pk(900, 1050), pk(1780, 1940), pk(1960, 2120)
a = (1332.492 - 661.657) / (c_co - c_cs)       # keV per channel
b = 661.657 - a * c_cs                          # keV at channel 0
print(c_cs, c_co, c_x, round(a, 4), round(b, 1), round(a * c_x + b))

In [ ]:
ch = np.array([0, 2200])
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([c_cs, c_co], [661.657, 1332.492], "o", ms=8, label="Cs-137 and Co-60 1332 (known)")
ax.plot(ch, a * ch + b, c="C0", label=f"E = {a:.4f}·ch {b:+.1f} keV")
ax.plot(ch, ch * 661.657 / c_cs, c="C3", ls="--", label="trap: line through the origin")
ax.plot([c_x, c_x, 0], [0, a * c_x + b, a * c_x + b], c="gray", ls=":")
ax.plot(c_x, a * c_x + b, "s", c="C2", label=f"unknown: channel {c_x} -> {a * c_x + b:.0f} keV")
ax.set_xlim(0, 2200); ax.set_ylim(0, 1700)
ax.set_xlabel("channel"); ax.set_ylabel("energy (keV)"); ax.legend(fontsize=8, loc="upper left"); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: independent line: the Co-60 1173 keV peak was not used for the line; where does it land?
c_co1 = pk(1600, 1760)
print(f"Co-60 1173 peak at channel {c_co1} -> {a * c_co1 + b:.0f} keV (true 1173.2 keV; one bin is {8 * a:.0f} keV wide)")
assert abs(a * c_co1 + b - 1173.228) < 8 * a
# Check B: magnitude: a gain near 0.75 keV per channel and an offset of tens of keV are typical of an ADC
print(f"gain {a:.4f} keV/channel, offset {b:.1f} keV")
assert 0.7 < a < 0.8 and -100 < b < 0
# trap: keV proportional to the channel (no offset)
print(f"trap: {c_x} x 661.657/{c_cs} = {c_x * 661.657 / c_cs:.0f} keV, which matches no line")

**Your turn.** Radon daughters give a Bi-214 line at 609.3 keV. With your calibration, at which channel would it appear? Is it far from the Cs-137 peak?

*Hint:* invert the line: ch = (E − b)/a.

In [ ]:
# TODO: write your answer here

## Problem 8 · A small bump above the background: is it real?
In a spectrum, the signal region around a small bump holds $N_{SR} = 1150$ counts. Two sidebands, together twice as wide as the signal region, hold $N_{SB} = 2000$ counts (illustrative). How many signal counts $S$ are there, and how many times their wobble is that?

*Given:* background in the signal region $\hat B = N_{SB}/2 = 1000$; every count wobbles by $\sqrt{N}$; $S = N_{SR} - \hat B$; wobble of $S$: $\sigma_S = \sqrt{N_{SR} + N_{SB}/4}$; $Z = S/\sigma_S$.

**Predict.** S and Z. Compute them by hand first. Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
N_SR, N_SB = 1150, 2000                # signal region, sidebands (twice as wide)
B = N_SB / 2
S, sS = N_SR - B, (N_SR + N_SB / 4)**0.5
print(S, round(sS, 1), round(S / sS, 1), round(S**0.5, 1))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(["N_SR (signal region)", "B̂ (background)", "S = N_SR − B̂"], [N_SR, B, S],
       yerr=[N_SR**0.5, (N_SB / 4)**0.5, sS], capsize=6, color=["C0", "C7", "C2"], alpha=0.7)
ax.set_ylabel("counts"); ax.set_title(f"S = {S:.0f} ± {sS:.1f} counts, Z = {S / sS:.1f}"); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: simple bound: the background estimate alone wobbles by sqrt(1000); even that allows at most 150/31.6
print(f"sqrt(B) = {B**0.5:.1f} counts -> S/sqrt(B) = {S / B**0.5:.1f}; trap sqrt(S) = {S**0.5:.1f} gives 'Z = {S / S**0.5:.1f}', which is impossible")
assert S / sS < S / B**0.5 < S / S**0.5
# Check B: units: S and sigma_S are both counts, so Z is a pure number
print(f"Z = {S:.0f} counts / {sS:.1f} counts = {S / sS:.1f}")

**Your turn.** If the bump is real, all counts grow in proportion to the measuring time. How many times longer must we measure to reach Z = 5?

*Hint:* S grows like t, its wobble like √t, so Z grows like √t.

In [ ]:
# TODO: write your answer here

## Before you close the tab
- Mistakes we met: an edge value counted twice (P1), the narrowest bins (P2), the mean with a glitch (P3), the mean as typical (P4), 1/n for a sample (P5), s as the error of the mean (P6), keV ∝ channel (P7), √S as the wobble (P8).
- Each has a check: counts add to N, √N bins, the median, the tallest bar, √(n/(n−1)), s/√n, a second known line, the background wobble.
- **Runtime → Restart session and run all**; **File → Save**. Files from this lab are in `MyDrive/PHY405_Labs/Week02`.
- Next week: random variables and probability distributions.